# Análisis experimental QSVM

El notebook reutiliza la preparación de datos y los feature maps de la clase `qsvm` para generar kernels ideales, calcular métricas y exportar todas las tablas y figuras a `img/` como PDF. Las figuras no incluyen títulos; cada archivo tiene un nombre descriptivo.

La sensibilidad al ruido usa un modelo analítico de despolarización global, documentado como aproximación reproducible. La sensibilidad al muestreo usa realizaciones binomiales de los valores exactos del kernel.

In [1]:
from pathlib import Path
from IPython.display import display

from qsvm_experiment_analysis import QSVMExperimentAnalysis


def encontrar_raiz_repo():
    actual = Path.cwd().resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "data" / "water_potability.csv").exists():
            return candidato
    raise FileNotFoundError("No se encontró la raíz del repositorio.")


REPO_ROOT = encontrar_raiz_repo()
IMG_DIR = REPO_ROOT / "img"
analisis = QSVMExperimentAnalysis(REPO_ROOT, IMG_DIR, random_seed=42)
print(f"PDFs: {IMG_DIR}")

PDFs: /Users/humberto/Documents/Git/Quantathon/img


## Combinaciones

In [2]:
COMBINACIONES = [
    {"N": 64, "dim": 5, "featuremap": "z_feature_map"},
    {"N": 48, "dim": 4, "featuremap": "zz_feature_map"},
    {"N": 48, "dim": 4, "featuremap": "custom_q_kernel"},
    {"N": 48, "dim": 6, "featuremap": "pauli_feature_map"},
    {"N": 24, "dim": 2, "featuremap": "pauli_feature_map"},
    {"N": 16, "dim": 2, "featuremap": "zz_feature_map"},
    {"N": 40, "dim": 2, "featuremap": "custom_q_kernel"},
    {"N": 72, "dim": 2, "featuremap": "z_feature_map"},
    {"N": 64, "dim": 2, "featuremap": "efficient_su2"},
]

COMBINACION_H2 = {"N": 40, "dim": 3, "featuremap": "efficient_su2"}

display(__import__("pandas").DataFrame(COMBINACIONES))

,N,dim,featuremap
0,64,5,z_feature_map
1,48,4,zz_feature_map
2,48,4,custom_q_kernel
3,48,6,pauli_feature_map
4,24,2,pauli_feature_map
5,16,2,zz_feature_map
6,40,2,custom_q_kernel
7,72,2,z_feature_map
8,64,2,efficient_su2


## Ejecución completa

In [3]:
resumen, faltantes_h2 = analisis.run(COMBINACIONES, COMBINACION_H2)
display(resumen)

if faltantes_h2:
    print("No se generó la comparación H2 porque faltan:")
    for archivo in faltantes_h2:
        print(f"  {archivo}")
else:
    print("Comparación H2 generada.")

Procesando n_64_dim_5_z_feature_map


Procesando n_48_dim_4_zz_feature_map


Procesando n_48_dim_4_custom_q_kernel


Procesando n_48_dim_6_pauli_feature_map


Procesando n_24_dim_2_pauli_feature_map


...

Procesando n_40_dim_2_custom_q_kernel


Procesando n_72_dim_2_z_feature_map


Procesando n_64_dim_2_efficient_su2


,N,dim,featuremap,accuracy,balanced_accuracy,f1,svc_seconds,cv_balanced_accuracy_mean,cv_balanced_accuracy_std,kernel_target_alignment,intra_class_similarity_mean,inter_class_similarity_mean,effective_rank,numerical_rank,circuit_depth,two_qubit_gates,gate_count,kernel_generation_seconds,train_kernel_circuits,test_kernel_circuits
0,64,5,z_feature_map,0.187500,0.214286,0.315789,0.000784,0.435714,0.087352,0.085009,0.029088,0.033682,51.018034,64,4,0,20,0.012207,2016,1024
1,48,4,zz_feature_map,0.666667,0.714286,0.714286,0.000451,0.495000,0.074833,0.096211,0.061130,0.067323,40.255797,48,16,8,28,0.013632,1128,576
2,48,4,custom_q_kernel,0.916667,0.900000,0.888889,0.000472,0.555000,0.097980,0.065632,0.143119,0.142947,19.834446,48,25,4,52,0.009556,1128,576
3,48,6,pauli_feature_map,0.416667,0.414286,0.363636,0.000621,0.625000,0.184391,0.146416,0.017178,0.014511,46.478485,48,24,12,54,0.028252,1128,576
4,24,2,pauli_feature_map,0.500000,0.625000,0.400000,0.000584,0.433333,0.097183,0.040892,0.244854,0.278573,8.310782,13,9,2,15,0.004629,276,144
5,16,2,zz_feature_map,0.750000,0.833333,0.666667,0.000554,0.650000,0.200000,0.148440,0.278197,0.253465,7.493097,13,7,2,11,0.002418,120,64
6,40,2,custom_q_kernel,0.900000,0.875000,0.857143,0.000631,0.611667,0.101462,0.040026,0.367051,0.360574,5.566927,9,6,1,12,0.005374,780,400
7,72,2,z_feature_map,0.388889,0.375000,0.266667,0.000665,0.433929,0.020825,0.041662,0.250243,0.239268,7.665491,9,4,0,8,0.006906,2556,1296
8,64,2,efficient_su2,0.437500,0.500000,0.608696,0.000624,0.445238,0.050843,0.001632,0.561826,0.574497,2.945189,4,10,0,10,0.006940,2016,1024


No se generó la comparación H2 porque faltan:
  /Users/humberto/Documents/Git/Quantathon/data/kernel_h2/n_40_dim_3_efficient_su2.csv
  /Users/humberto/Documents/Git/Quantathon/data/test_h2/n_40_dim_3_efficient_su2.csv


## Salidas

Por combinación se exportan: métricas y costo; mapa de calor y distribución del kernel; similitudes intra/interclase; espectro; circuitos original y transpilado; sensibilidad al muestreo y al ruido; y ablaciones de repeticiones, entrelazamiento, términos de Pauli, escalado y recarga de datos.

Para `n_40_dim_3_efficient_su2`, cuando existan los CSV locales y H2, se exportan mapas de entrenamiento y prueba, diferencia con signo, error cuadrado y `sqrt(error cuadrado)`, equivalente al error absoluto.